# TSP Art with SAM and LKH

This notebook is a full English walkthrough version of `LKH-3_SAM_colored_connected_parallel.py`.

It contains the complete code from the current script, split into logical sections, with a short explanation before each block. You can read it as documentation or run it cell by cell as a notebook version of the same pipeline.


## 1. Imports, optional dependencies, and path defaults

This section loads the libraries used by the project and initializes portable path defaults.

Important points:
- `opencv-python` is optional and is used to split masks into connected components.
- `segment-anything` is optional until you actually generate SAM masks.
- `SCRIPT_DIR` allows the code to use project-relative defaults instead of a machine-specific absolute path.


In [ ]:
from __future__ import annotations

import argparse
import json
import logging
import os
import subprocess
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
from PIL import Image, ImageEnhance

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection


SCRIPT_DIR = Path(__file__).resolve().parent


# ============================================================
# Optional dependency: OpenCV for connected components
#   pip install opencv-python
# ============================================================

try:
    import cv2  # type: ignore
except Exception:
    cv2 = None


# ============================================================
# NOTE: SAM dependency
#   pip install opencv-python
#   then clone segment-anything:
#       git clone https://github.com/facebookresearch/segment-anything
#       pip install -e segment-anything
#   download ckpt (e.g. sam_vit_h_4b8939.pth)
# ============================================================

try:
    from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
except Exception as e:
    sam_model_registry = None
    SamAutomaticMaskGenerator = None
    _SAM_IMPORT_ERROR = e


# ==========================
# Defaults
# ==========================
DEFAULT_LKH_EXE = os.environ.get("LKH_EXE", str(SCRIPT_DIR / "LKH-3.exe"))
DEFAULT_OUTPUT_DIR = str(SCRIPT_DIR / "outputs")


## 2. Logging and run-folder management

These helpers make each run reproducible and easier to inspect.

- `setup_logger` writes both console and file logs.
- `make_run_dir` creates a timestamped output folder for a run.
- `save_config` stores the exact resolved configuration as JSON.


In [ ]:
# ==========================
# Logging
# ==========================
def setup_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger(f"tsp_art_{log_path.parent.name}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter("[%(asctime)s] %(levelname)s: %(message)s", "%Y-%m-%d %H:%M:%S")

    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setLevel(logging.INFO)
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    return logger


def make_run_dir(base_output_dir: Path, name_hint: str) -> Path:
    ts = time.strftime("%Y%m%d_%H%M%S")
    run_dir = base_output_dir / "runs" / f"{ts}_{name_hint}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


def save_config(cfg, out_path: Path) -> None:
    out_path.write_text(json.dumps(asdict(cfg), indent=2, ensure_ascii=False), encoding="utf-8")


## 3. Central experiment configuration

The entire pipeline is configured through one dataclass.

This is useful because the same parameter object can be created from notebook code or from command-line arguments.


In [ ]:
# ==========================
# Config
# ==========================
@dataclass
class ExperimentConfig:
    image_path: str
    output_dir: str
    lkh_exe: str

    # base image preprocessing
    target_width: int = 800
    contrast: float = 1.4
    gamma: float = 1.0
    clip_percent: float = 1.0

    # density + dithering
    density_mode: str = "grad"  # gray | inv | grad | sat | mix_gray_grad
    black_quantile: float = 0.55
    border_margin_px: int = 0  # drop cities within this many pixels of the image border
    use_adaptive_threshold: bool = True

    mix_gray_weight: float = 0.60
    mix_grad_weight: float = 0.40

    # SAM
    sam_model_type: str = "vit_h"   # vit_h | vit_l | vit_b
    sam_ckpt: str = ""
    sam_device: str = "cuda"        # cuda | cpu
    sam_points_per_side: int = 32   # higher -> more masks, slower
    sam_pred_iou_thresh: float = 0.88
    sam_stability_score_thresh: float = 0.92
    sam_min_mask_region_area: int = 0  # SAM internal post-process

    # layer selection
    max_layers: int = 30
    min_layer_area_ratio: float = 0.005  # ignore tiny masks (<0.5% of image)
    nonoverlap: bool = True              # subtract used pixels to reduce overdraw
    add_background_layer: bool = True

    # per-layer city budget
    max_cities_total: int = 30000
    min_cities_per_layer: int = 600
    max_cities_per_layer: int = 12000
    seed: int = 42

    # LKH
    runs: int = 5
    time_limit: int = 300
    max_trials: int = 200
    trace_level: int = 1
    lkh_workers: int = max(1, (os.cpu_count() or 4) // 2)

    # render
    linewidth: float = 0.6
    layer_alpha: float = 0.75
    dpi: int = 300

    # debug
    save_layer_previews: bool = True
    preview_only: bool = False  # only produce masks + previews, no LKH


## 4. Image helpers and density-field construction

This section converts the source image into grayscale and other derived representations.

The code builds several density fields:
- `gray`: raw grayscale
- `inv`: inverted grayscale
- `grad`: inverted edge magnitude
- `sat`: inverted color saturation
- `mix_gray_grad`: weighted mixture of grayscale and gradient

Later, one of these fields is selected to drive dithering and point placement.


In [ ]:
# ==========================
# Image helpers
# ==========================
def load_and_resize_rgb(image_path: str, target_width: int) -> np.ndarray:
    img = Image.open(image_path).convert("RGB")
    aspect = img.height / img.width
    target_height = int(target_width * aspect)
    img = img.resize((target_width, target_height), Image.LANCZOS)
    return np.array(img, dtype=np.uint8)


def rgb_to_gray_u8(rgb_u8: np.ndarray) -> np.ndarray:
    return np.array(Image.fromarray(rgb_u8).convert("L"), dtype=np.uint8)


def preprocess_gray_u8(gray_u8: np.ndarray, contrast: float, gamma: float, clip_percent: float) -> np.ndarray:
    img = Image.fromarray(gray_u8).convert("L")
    img = ImageEnhance.Contrast(img).enhance(contrast)
    g = np.array(img, dtype=np.uint8).astype(np.float32)

    if clip_percent is not None and clip_percent > 0:
        lo = np.percentile(g, clip_percent)
        hi = np.percentile(g, 100 - clip_percent)
        if hi > lo + 1e-6:
            g = (g - lo) * (255.0 / (hi - lo))
            g = np.clip(g, 0, 255)

    if gamma is not None and abs(gamma - 1.0) > 1e-6:
        x = np.clip(g / 255.0, 0, 1)
        g = (x ** (1.0 / gamma)) * 255.0
        g = np.clip(g, 0, 255)

    return g.astype(np.uint8)


def to_u8_0_255(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    mn, mx = float(np.min(x)), float(np.max(x))
    if mx <= mn + 1e-6:
        return np.zeros_like(x, dtype=np.uint8)
    y = (x - mn) / (mx - mn)
    return np.clip(y * 255.0, 0, 255).astype(np.uint8)


def grad_magnitude(gray_u8: np.ndarray) -> np.ndarray:
    g = gray_u8.astype(np.float32)
    gx = np.zeros_like(g)
    gy = np.zeros_like(g)
    gx[:, 1:-1] = (g[:, 2:] - g[:, :-2]) * 0.5
    gy[1:-1, :] = (g[2:, :] - g[:-2, :]) * 0.5
    return np.sqrt(gx * gx + gy * gy)


def saturation_from_rgb(rgb_u8: np.ndarray) -> np.ndarray:
    rgb = rgb_u8.astype(np.float32) / 255.0
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    mx = np.maximum(np.maximum(r, g), b)
    mn = np.minimum(np.minimum(r, g), b)
    sat = np.where(mx < 1e-6, 0.0, (mx - mn) / (mx + 1e-6))
    return sat


def build_density_fields(
    gray_u8: np.ndarray,
    rgb_u8: np.ndarray,
    mix_gray_weight: float,
    mix_grad_weight: float,
) -> Dict[str, np.ndarray]:
    fields: Dict[str, np.ndarray] = {}
    fields["gray"] = gray_u8.copy()
    fields["inv"] = (255 - gray_u8).astype(np.uint8)

    mag = grad_magnitude(gray_u8)
    mag_u8 = to_u8_0_255(mag)
    fields["grad"] = (255 - mag_u8).astype(np.uint8)

    sat = saturation_from_rgb(rgb_u8)
    sat_u8 = to_u8_0_255(sat)
    fields["sat"] = (255 - sat_u8).astype(np.uint8)

    w1, w2 = float(mix_gray_weight), float(mix_grad_weight)
    s = max(w1 + w2, 1e-6)
    w1, w2 = w1 / s, w2 / s
    mix = w1 * fields["gray"].astype(np.float32) + w2 * fields["grad"].astype(np.float32)
    fields["mix_gray_grad"] = np.clip(mix, 0, 255).astype(np.uint8)

    return fields


def save_u8_png(arr_u8: np.ndarray, path: Path) -> None:
    Image.fromarray(arr_u8).save(path)


## 5. Floyd-Steinberg dithering

The selected density field is turned into a binary image. Black pixels are later interpreted as cities.

The advantage of Floyd-Steinberg dithering is that it preserves visual tone through spatial error diffusion, which gives a better point distribution than plain thresholding.


In [ ]:
# ==========================
# Dithering
# ==========================
def floyd_steinberg_from_u8(input_u8: np.ndarray, black_quantile: float, use_adaptive_threshold: bool) -> np.ndarray:
    pixels = input_u8.astype(np.float32)
    h, w = pixels.shape
    thresh = float(np.quantile(pixels, black_quantile)) if use_adaptive_threshold else 160.0

    for y in range(h):
        for x in range(w):
            old = pixels[y, x]
            new = 255.0 if old > thresh else 0.0
            pixels[y, x] = new
            err = old - new

            if x + 1 < w:
                pixels[y, x + 1] += err * 7 / 16
            if x - 1 >= 0 and y + 1 < h:
                pixels[y + 1, x - 1] += err * 3 / 16
            if y + 1 < h:
                pixels[y + 1, x] += err * 5 / 16
            if x + 1 < w and y + 1 < h:
                pixels[y + 1, x + 1] += err * 1 / 16

    return pixels.astype(np.uint8)


## 6. Converting dithered pixels into TSP cities

Once the image is dithered, black pixels become candidate city coordinates.

This block also supports:
- optional removal of border-near cities
- random subsampling when the number of points exceeds the budget
- a preview image for debugging city placement


In [ ]:
# ==========================
# Cities
# ==========================
def extract_cities_from_dithered(
    dithered_u8: np.ndarray,
    max_cities: int,
    seed: int,
    border_margin_px: int = 0,
) -> np.ndarray:
    """
    Extract city coordinates (x,y) from a binary dithered image where black pixels are cities.

    border_margin_px:
        If > 0, drop any cities within `border_margin_px` of the image boundary.
        This is a strict, cheap way to eliminate edge outliers that often cause very long TSP edges.
    """
    h, w = dithered_u8.shape[:2]
    ys, xs = np.where(dithered_u8 == 0)

    if len(xs) == 0:
        return np.zeros((0, 2), dtype=int)

    m = int(border_margin_px)
    if m > 0:
        keep = (xs >= m) & (xs < (w - m)) & (ys >= m) & (ys < (h - m))
        xs = xs[keep]
        ys = ys[keep]
        if len(xs) == 0:
            return np.zeros((0, 2), dtype=int)

    cities = np.stack([xs, ys], axis=1).astype(int)

    if len(cities) > max_cities:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(cities), max_cities, replace=False)
        cities = cities[idx]

    return cities


def save_cities_preview(cities: np.ndarray, w: int, h: int, out_path: Path) -> None:
    canvas = np.full((h, w), 255, dtype=np.uint8)
    if len(cities) > 0:
        xs = np.clip(cities[:, 0], 0, w - 1)
        ys = np.clip(cities[:, 1], 0, h - 1)
        canvas[ys, xs] = 0
    Image.fromarray(canvas).save(out_path)


## 7. SAM masks, layer selection, connected components, and color sampling

This part creates structural layers from the source image.

Key ideas:
- generate candidate masks with Segment Anything
- keep only masks above a minimum area ratio
- optionally enforce non-overlap so each pixel is claimed once
- optionally add a background layer
- split each layer into connected components to avoid long cross-image TSP edges
- sample per-segment colors from the original image for richer rendering


In [ ]:
# ==========================
# SAM masks
# ==========================
def sam_generate_masks(
    rgb_u8: np.ndarray,
    model_type: str,
    ckpt_path: str,
    device: str,
    points_per_side: int,
    pred_iou_thresh: float,
    stability_score_thresh: float,
    min_mask_region_area: int,
    logger: logging.Logger,
) -> List[dict]:
    if sam_model_registry is None or SamAutomaticMaskGenerator is None:
        raise RuntimeError(
            "segment-anything is not installed/importable.\n"
            f"Import error: {_SAM_IMPORT_ERROR}\n"
            "Install: pip install -e segment-anything (after cloning repo)"
        )

    if not ckpt_path or not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"SAM checkpoint not found: {ckpt_path}")

    logger.info(f"Loading SAM: {model_type} on {device}")
    sam = sam_model_registry[model_type](checkpoint=ckpt_path)
    sam.to(device=device)

    gen = SamAutomaticMaskGenerator(
        model=sam,
        points_per_side=int(points_per_side),
        pred_iou_thresh=float(pred_iou_thresh),
        stability_score_thresh=float(stability_score_thresh),
        min_mask_region_area=int(min_mask_region_area),
    )

    logger.info("Generating SAM masks...")
    masks = gen.generate(rgb_u8)  # list[dict], each has 'segmentation' (bool HxW), 'area', 'bbox', ...
    logger.info(f"SAM masks generated: {len(masks)}")
    return masks


def select_layers_from_masks(
    masks: List[dict],
    h: int,
    w: int,
    max_layers: int,
    min_area_ratio: float,
    nonoverlap: bool,
    add_background_layer: bool,
    logger: logging.Logger,
) -> List[np.ndarray]:
    img_area = h * w
    min_area = int(min_area_ratio * img_area)

    # Filter by area
    kept = [m for m in masks if int(m.get("area", 0)) >= min_area]
    logger.info(f"Masks after area filter (>= {min_area} px): {len(kept)}")

    # Sort: small-to-large helps detail-first when nonoverlap is enabled
    kept.sort(key=lambda m: int(m.get("area", 0)))

    layers: List[np.ndarray] = []
    uncovered = np.ones((h, w), dtype=bool)

    for m in kept:
        seg = m["segmentation"].astype(bool)
        if nonoverlap:
            seg = seg & uncovered
        area = int(seg.sum())
        if area < min_area:
            continue

        layers.append(seg)
        if nonoverlap:
            uncovered[seg] = False

        if len(layers) >= max_layers:
            break

    if add_background_layer:
        bg = uncovered.copy() if nonoverlap else None
        # If nonoverlap=False, "background" is ambiguous; we can still add "not covered by selected layers"
        if not nonoverlap:
            covered = np.zeros((h, w), dtype=bool)
            for seg in layers:
                covered |= seg
            bg = ~covered
        if bg is not None and int(bg.sum()) > 0:
            layers.append(bg)

    logger.info(f"Final layers count: {len(layers)}")
    return layers


def robust_layer_color(rgb_u8: np.ndarray, mask: np.ndarray) -> Tuple[int, int, int]:
    """Median RGB (more robust than mean)."""
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return (0, 0, 0)
    pix = rgb_u8[ys, xs, :]  # (N,3)
    med = np.median(pix, axis=0)
    return (int(med[0]), int(med[1]), int(med[2]))


def split_mask_connected_components(mask: np.ndarray, min_area_px: int) -> List[np.ndarray]:
    """Split a boolean mask into connected components (8-connectivity).

    This is used to ensure each solved TSP layer is *single connected*, which
    greatly reduces long "cross-image" edges caused by a layer containing
    multiple disjoint islands.
    """
    if mask.dtype != bool:
        mask = mask.astype(bool)
    if int(mask.sum()) <= 0:
        return []

    if cv2 is None:
        # Fallback: no splitting (still works, but may create long edges)
        return [mask]

    m = (mask.astype(np.uint8) * 255)
    num, labels = cv2.connectedComponents(m, connectivity=8)
    if num <= 2:
        return [mask]

    comps: List[np.ndarray] = []
    # label 0 is background
    for k in range(1, num):
        comp = labels == k
        if int(comp.sum()) >= int(min_area_px):
            comps.append(comp)

    # largest-first is convenient for directory order/readability
    comps.sort(key=lambda x: int(x.sum()), reverse=True)
    return comps


def segment_colors_from_midpoints(rgb_u8: np.ndarray, pts_xy: np.ndarray, alpha: float) -> np.ndarray:
    """Assign a color to each tour segment by sampling the original RGB image.

    We sample at the midpoint of each segment (point i -> point i+1). This produces
    a much more vivid, source-faithful coloring than using a single layer-mean color.

    Returns (N,4) float32 RGBA in [0,1], where N=len(pts_xy).
    """
    h, w, _ = rgb_u8.shape
    p0 = pts_xy.astype(np.float32)
    p1 = np.roll(p0, -1, axis=0)
    mids = ((p0 + p1) * 0.5).astype(np.int32)
    mx = np.clip(mids[:, 0], 0, w - 1)
    my = np.clip(mids[:, 1], 0, h - 1)
    c = rgb_u8[my, mx].astype(np.float32) / 255.0
    a = np.full((len(c), 1), float(alpha), dtype=np.float32)
    return np.concatenate([c, a], axis=1)


## 8. TSPLIB writing, LKH execution, and tour parsing

LKH works with TSPLIB-style files, so the point cloud must be written to disk before solving.

This section writes the problem and parameter files, runs `LKH-3.exe`, parses the generated tour, and exposes a worker function for parallel execution.


In [ ]:
# ==========================
# LKH (TSPLIB EUC_2D)
# ==========================
def write_tsplib_euc2d(cities: np.ndarray, tsp_path: Path, name: str) -> None:
    n = len(cities)
    lines = [
        f"NAME : {name}",
        "TYPE : TSP",
        f"DIMENSION : {n}",
        "EDGE_WEIGHT_TYPE : EUC_2D",
        "NODE_COORD_SECTION",
    ]
    for i, (x, y) in enumerate(cities, start=1):
        lines.append(f"{i} {int(x)} {int(y)}")
    lines.append("EOF")
    tsp_path.write_text("\n".join(lines), encoding="utf-8")


def write_lkh_par(
    par_path: Path,
    tsp_path: Path,
    tour_path: Path,
    runs: int,
    seed: int,
    trace_level: int,
    time_limit: int,
    max_trials: int,
) -> None:
    lines = [
        f"PROBLEM_FILE = {tsp_path}",
        f"OUTPUT_TOUR_FILE = {tour_path}",
        f"RUNS = {int(runs)}",
        f"SEED = {int(seed)}",
        f"TRACE_LEVEL = {int(trace_level)}",
        f"TIME_LIMIT = {int(time_limit)}",
        f"MAX_TRIALS = {int(max_trials)}",
    ]
    par_path.write_text("\n".join(lines), encoding="utf-8")


def read_lkh_tour(tour_path: Path) -> np.ndarray:
    txt = tour_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    start = None
    for k, line in enumerate(txt):
        if line.strip() == "TOUR_SECTION":
            start = k + 1
            break
    if start is None:
        raise RuntimeError("TOUR_SECTION not found.")

    tour = []
    for line in txt[start:]:
        s = line.strip()
        if not s:
            continue
        if s == "-1" or s.upper() == "EOF":
            break
        tour.append(int(s))

    if not tour:
        raise RuntimeError("Empty tour.")
    return np.array(tour, dtype=int) - 1


def solve_tsp_lkh(
    cities: np.ndarray,
    lkh_exe: str,
    work_dir: Path,
    runs: int,
    seed: int,
    time_limit: int,
    max_trials: int,
    trace_level: int,
    logger: logging.Logger,
) -> np.ndarray:
    lkh_exe_abs = os.path.abspath(lkh_exe)
    if not os.path.exists(lkh_exe_abs):
        raise FileNotFoundError(
            f"LKH executable not found: {lkh_exe_abs}\n"
            "Pass --lkh_exe or set env var LKH_EXE."
        )

    tsp_path = work_dir / "problem.tsp"
    par_path = work_dir / "problem.par"
    tour_path = work_dir / "problem.tour"

    write_tsplib_euc2d(cities, tsp_path, name="tsp_layer")
    write_lkh_par(par_path, tsp_path, tour_path, runs, seed, trace_level, time_limit, max_trials)

    logger.info(f"Running LKH... (N={len(cities)}, runs={runs}, tlim={time_limit})")
    subprocess.run([lkh_exe_abs, str(par_path)], cwd=str(work_dir), check=True)

    if not tour_path.exists():
        raise RuntimeError("LKH did not produce tour file.")
    return read_lkh_tour(tour_path)



def _lkh_worker(job: Dict) -> Tuple[np.ndarray, np.ndarray]:
    """Run one LKH instance in a separate process and return (cities, tour).

    Each job writes/reads only inside its own work_dir, so it's safe to run many in parallel.
    """
    work_dir = Path(job["work_dir"])

    # Use a local logger to avoid cross-process file handler contention.
    worker_logger = logging.getLogger(f"lkh_worker_{os.getpid()}")
    worker_logger.handlers.clear()
    worker_logger.setLevel(logging.ERROR)

    tour = solve_tsp_lkh(
        cities=job["cities"],
        lkh_exe=job["lkh_exe"],
        work_dir=work_dir,
        runs=int(job["runs"]),
        seed=int(job["seed"]),
        time_limit=int(job["time_limit"]),
        max_trials=int(job["max_trials"]),
        trace_level=int(job["trace_level"]),
        logger=worker_logger,
    )
    return job["cities"], tour


## 9. Rendering the final colored line drawing

After each layer or connected component has a valid tour, the code renders all tours into one figure.

The rendering uses `LineCollection`, and each segment color is sampled from the original image at the segment midpoint. This gives a more image-faithful result than using one average color per layer.


In [ ]:
# ==========================
# Rendering
# ==========================
def render_layers_colored(
    layer_results: List[Tuple[np.ndarray, np.ndarray]],  # (cities, tour)
    rgb_u8: np.ndarray,
    w: int,
    h: int,
    out_path: Path,
    linewidth: float,
    alpha: float,
    dpi: int,
    logger: logging.Logger,
) -> None:
    logger.info("Rendering final layered colored drawing...")

    fig, ax = plt.subplots(1, 1, figsize=(12, 12), dpi=100)

    for cities, tour in layer_results:
        if len(cities) < 2:
            continue
        pts = cities[tour]  # (N,2)
        segs = np.stack([pts, np.roll(pts, -1, axis=0)], axis=1).astype(np.float32)

        # Color each segment by sampling the original RGB image.
        # This dramatically improves color diversity vs a single "layer-average" color.
        seg_colors = segment_colors_from_midpoints(rgb_u8=rgb_u8, pts_xy=pts, alpha=alpha)
        lc = LineCollection(segs, colors=seg_colors, linewidths=linewidth)
        ax.add_collection(lc)

    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.set_aspect("equal")
    ax.axis("off")

    plt.tight_layout()
    plt.savefig(out_path, dpi=dpi, bbox_inches="tight", pad_inches=0, facecolor="white")
    plt.close()
    logger.info(f"Saved: {out_path.name}")


## 10. Main pipeline: SAM layers to final artwork

This is the core function of the project.

`run_sam_layers` performs the full workflow: load the image, generate density maps, build SAM layers, extract point sets, run LKH in parallel, and render the final colored drawing.


In [ ]:
# ==========================
# Main pipeline: SAM layers
# ==========================
def run_sam_layers(cfg: ExperimentConfig) -> None:
    base_out = Path(cfg.output_dir)
    base_out.mkdir(parents=True, exist_ok=True)

    name_hint = f"sam_layers_w{cfg.target_width}_K{cfg.max_layers}_{cfg.density_mode}"
    run_dir = make_run_dir(base_out, name_hint=name_hint)
    logger = setup_logger(run_dir / "run.log")
    save_config(cfg, run_dir / "config.json")

    logger.info("=" * 80)
    logger.info("TSP Art (SAM layers): mask -> masked density -> dithering -> cities -> LKH -> colored overlay")
    logger.info(f"Run dir: {run_dir}")
    logger.info("=" * 80)

    # 0) load base image
    rgb = load_and_resize_rgb(cfg.image_path, cfg.target_width)
    h, w, _ = rgb.shape
    gray = preprocess_gray_u8(rgb_to_gray_u8(rgb), cfg.contrast, cfg.gamma, cfg.clip_percent)

    Image.fromarray(rgb).save(run_dir / "preview_resized_rgb.png")
    save_u8_png(gray, run_dir / "preview_resized_gray.png")

    # 1) density base
    densities = build_density_fields(gray, rgb, cfg.mix_gray_weight, cfg.mix_grad_weight)
    if cfg.density_mode not in densities:
        raise ValueError(f"Unknown density_mode={cfg.density_mode}. Available: {list(densities.keys())}")
    base_density = densities[cfg.density_mode]

    if cfg.save_layer_previews:
        for k, den in densities.items():
            save_u8_png(den, run_dir / f"preview_density_{k}.png")

    # 2) SAM masks
    masks = sam_generate_masks(
        rgb_u8=rgb,
        model_type=cfg.sam_model_type,
        ckpt_path=cfg.sam_ckpt,
        device=cfg.sam_device,
        points_per_side=cfg.sam_points_per_side,
        pred_iou_thresh=cfg.sam_pred_iou_thresh,
        stability_score_thresh=cfg.sam_stability_score_thresh,
        min_mask_region_area=cfg.sam_min_mask_region_area,
        logger=logger,
    )

    layers = select_layers_from_masks(
        masks=masks,
        h=h,
        w=w,
        max_layers=cfg.max_layers,
        min_area_ratio=cfg.min_layer_area_ratio,
        nonoverlap=cfg.nonoverlap,
        add_background_layer=cfg.add_background_layer,
        logger=logger,
    )

    # 3) allocate per-layer city budgets
    layer_areas = np.array([int(m.sum()) for m in layers], dtype=np.int64)
    total_area = int(layer_areas.sum()) if int(layer_areas.sum()) > 0 else 1

    budgets: List[int] = []
    for area in layer_areas:
        raw = int(cfg.max_cities_total * (area / total_area))
        b = int(np.clip(raw, cfg.min_cities_per_layer, cfg.max_cities_per_layer))
        budgets.append(b)

    logger.info("Layer budgets (cities cap) summary:")
    logger.info(f"  layers={len(layers)}, total_city_budget={cfg.max_cities_total}")
    logger.info(f"  per-layer cap min={min(budgets)}, max={max(budgets)}, avg={sum(budgets)/len(budgets):.1f}")

    # 4) per-layer preview + (optional) solve
    layer_results: List[Tuple[np.ndarray, np.ndarray]] = []
    lkh_jobs: List[Dict] = []
    rng_seed_base = int(cfg.seed)

    layers_dir = run_dir / "layers"
    layers_dir.mkdir(exist_ok=True)

    for i, mask in enumerate(layers):
        layer_dir = layers_dir / f"layer_{i:03d}"
        layer_dir.mkdir(exist_ok=True)

        area = int(mask.sum())
        cap = budgets[i]

        # Ensure single connectivity: split disjoint islands into separate components.
        # Components that are too small are discarded.
        min_comp_area = max(int(cfg.min_layer_area_ratio * h * w * 0.5), 64)
        comps = split_mask_connected_components(mask, min_area_px=min_comp_area)
        if len(comps) == 0:
            continue

        # Distribute the layer city cap across components (area-proportional).
        comp_areas = np.array([int(c.sum()) for c in comps], dtype=np.int64)
        comp_total = int(comp_areas.sum()) if int(comp_areas.sum()) > 0 else 1
        comp_caps: List[int] = []
        for a in comp_areas:
            raw = int(cap * (a / comp_total))
            # components can be smaller; don't enforce the global per-layer minimum too aggressively
            b = int(np.clip(raw, 200, cap))
            comp_caps.append(b)

        if len(comps) > 1:
            logger.info(f"[Layer {i:03d}] split into {len(comps)} connected components (min_area_px={min_comp_area}).")

        # Process each connected component as an independent "sub-layer".
        for j, (comp_mask, comp_cap) in enumerate(zip(comps, comp_caps)):
            comp_dir = layer_dir if len(comps) == 1 else (layer_dir / f"comp_{j:02d}")
            comp_dir.mkdir(exist_ok=True)

            comp_area = int(comp_mask.sum())

            if cfg.save_layer_previews:
                mask_u8 = (comp_mask.astype(np.uint8) * 255)
                save_u8_png(mask_u8, comp_dir / "mask.png")

            # masked density: outside mask => 255
            masked_density = np.where(comp_mask, base_density, 255).astype(np.uint8)
            if cfg.save_layer_previews:
                save_u8_png(masked_density, comp_dir / "masked_density.png")

            # dither
            dithered = floyd_steinberg_from_u8(masked_density, cfg.black_quantile, cfg.use_adaptive_threshold)
            if cfg.save_layer_previews:
                save_u8_png(dithered, comp_dir / "dithered.png")

            # cities
            seed_ij = rng_seed_base + i * 1000 + j
            cities = extract_cities_from_dithered(dithered, max_cities=int(comp_cap), seed=seed_ij, border_margin_px=cfg.border_margin_px)
            if cfg.save_layer_previews:
                save_cities_preview(cities, w=w, h=h, out_path=comp_dir / "cities.png")

            logger.info(
                f"[Layer {i:03d}.{j:02d}] area={comp_area} cap={int(comp_cap)} cities={len(cities)}"
            )

            if cfg.preview_only:
                continue

            if len(cities) < 100:
                logger.info(f"[Layer {i:03d}.{j:02d}] skipped (too few cities).")
                continue

            # Defer LKH to a process pool so multiple components run in parallel.
            lkh_jobs.append({
                "cities": cities,
                "lkh_exe": cfg.lkh_exe,
                "work_dir": str(comp_dir),
                "runs": cfg.runs,
                "seed": seed_ij,
                "time_limit": cfg.time_limit,
                "max_trials": cfg.max_trials,
                "trace_level": cfg.trace_level,
                "layer_i": i,
                "comp_j": j,
            })

    # 4.9) Run LKH jobs in parallel (process pool)
    if not cfg.preview_only:
        if len(lkh_jobs) == 0:
            logger.warning("No LKH jobs to run (all layers/components were skipped).")
        else:
            max_workers = int(cfg.lkh_workers) if int(cfg.lkh_workers) > 0 else 1
            logger.info(f"Running {len(lkh_jobs)} LKH jobs with lkh_workers={max_workers} ...")

            results_with_keys: List[Tuple[Tuple[int, int], Tuple[np.ndarray, np.ndarray]]] = []
            with ProcessPoolExecutor(max_workers=max_workers) as ex:
                fut_map = {ex.submit(_lkh_worker, job): (int(job["layer_i"]), int(job["comp_j"])) for job in lkh_jobs}
                for fut in as_completed(fut_map):
                    key = fut_map[fut]
                    cities, tour = fut.result()
                    results_with_keys.append((key, (cities, tour)))

            # Stable ordering: (layer index, component index)
            results_with_keys.sort(key=lambda x: x[0])
            layer_results.extend([ct for _, ct in results_with_keys])

            logger.info(f"LKH finished: {len(layer_results)} tours collected.")

    # 5) final render
    if cfg.preview_only:
        logger.info("preview_only=True: done after generating masks/previews.")
        logger.info(f"Outputs: {run_dir}")
        return

    out_path = run_dir / "line_drawing_layers.png"
    render_layers_colored(
        layer_results=layer_results,
        rgb_u8=rgb,
        w=w,
        h=h,
        out_path=out_path,
        linewidth=cfg.linewidth,
        alpha=cfg.layer_alpha,
        dpi=cfg.dpi,
        logger=logger,
    )

    logger.info("=" * 80)
    logger.info(f"COMPLETE. Outputs: {run_dir}")
    logger.info("=" * 80)


## 11. Command-line interface

The original file is also a CLI script.

`build_argparser` defines the full argument surface, and `main` converts those arguments into `ExperimentConfig` before calling `run_sam_layers`.


In [ ]:
# ==========================
# CLI
# ==========================
def build_argparser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="TSP Art (SAM layers): mask -> per-layer TSP -> colored overlay")

    p.add_argument("--image", dest="image_path", required=True, help="Input image path")
    p.add_argument("--output_dir", default=DEFAULT_OUTPUT_DIR, help="Output base directory")
    p.add_argument("--lkh_exe", default=DEFAULT_LKH_EXE, help="Path to LKH executable (or env LKH_EXE)")

    # image preprocessing
    p.add_argument("--target_width", type=int, default=800)
    p.add_argument("--contrast", type=float, default=1.4)
    p.add_argument("--gamma", type=float, default=1.0)
    p.add_argument("--clip_percent", type=float, default=1.0)

    # density
    p.add_argument("--density_mode", type=str, default="grad", help="gray|inv|grad|sat|mix_gray_grad")
    p.add_argument("--black_quantile", type=float, default=0.55)
    p.add_argument("--border_margin_px", type=int, default=0, help="Drop cities within this many pixels of the image border (strict outlier removal). 0 disables.")
    p.add_argument("--no_adaptive_threshold", action="store_true")
    p.add_argument("--mix_gray_weight", type=float, default=0.60)
    p.add_argument("--mix_grad_weight", type=float, default=0.40)

    # SAM
    p.add_argument("--sam_model_type", type=str, default="vit_h")
    p.add_argument("--sam_ckpt", type=str, required=True, help="Path to SAM checkpoint .pth")
    p.add_argument("--sam_device", type=str, default="cuda")
    p.add_argument("--sam_points_per_side", type=int, default=32)
    p.add_argument("--sam_pred_iou_thresh", type=float, default=0.88)
    p.add_argument("--sam_stability_score_thresh", type=float, default=0.92)
    p.add_argument("--sam_min_mask_region_area", type=int, default=0)

    # layers
    p.add_argument("--max_layers", type=int, default=30)
    p.add_argument("--min_layer_area_ratio", type=float, default=0.005)
    p.add_argument("--nonoverlap", action="store_true", help="Subtract used pixels to reduce overlap")
    p.add_argument("--no_background_layer", action="store_true")

    # city budgets
    p.add_argument("--max_cities_total", type=int, default=30000)
    p.add_argument("--min_cities_per_layer", type=int, default=600)
    p.add_argument("--max_cities_per_layer", type=int, default=12000)
    p.add_argument("--seed", type=int, default=42)

    # LKH
    p.add_argument("--runs", type=int, default=5)
    p.add_argument("--time_limit", type=int, default=300)
    p.add_argument("--max_trials", type=int, default=200)
    p.add_argument("--trace_level", type=int, default=1)
    p.add_argument("--lkh_workers", type=int, default=max(1, (os.cpu_count() or 4) // 2),
                   help="Number of CPU processes to run LKH in parallel. Suggest: ~half your logical cores.")

    # render
    p.add_argument("--linewidth", type=float, default=0.6)
    p.add_argument("--layer_alpha", type=float, default=0.75)
    p.add_argument("--dpi", type=int, default=300)

    # debug
    p.add_argument("--no_layer_previews", action="store_true")
    p.add_argument("--preview_only", action="store_true")

    return p


def main() -> None:
    args = build_argparser().parse_args()

    cfg = ExperimentConfig(
        image_path=args.image_path,
        output_dir=args.output_dir,
        lkh_exe=args.lkh_exe,

        target_width=args.target_width,
        contrast=args.contrast,
        gamma=args.gamma,
        clip_percent=args.clip_percent,

        density_mode=args.density_mode,
        black_quantile=args.black_quantile,
        border_margin_px=args.border_margin_px,
        use_adaptive_threshold=(not args.no_adaptive_threshold),
        mix_gray_weight=args.mix_gray_weight,
        mix_grad_weight=args.mix_grad_weight,

        sam_model_type=args.sam_model_type,
        sam_ckpt=args.sam_ckpt,
        sam_device=args.sam_device,
        sam_points_per_side=args.sam_points_per_side,
        sam_pred_iou_thresh=args.sam_pred_iou_thresh,
        sam_stability_score_thresh=args.sam_stability_score_thresh,
        sam_min_mask_region_area=args.sam_min_mask_region_area,

        max_layers=args.max_layers,
        min_layer_area_ratio=args.min_layer_area_ratio,
        nonoverlap=bool(args.nonoverlap),
        add_background_layer=(not args.no_background_layer),

        max_cities_total=args.max_cities_total,
        min_cities_per_layer=args.min_cities_per_layer,
        max_cities_per_layer=args.max_cities_per_layer,
        seed=args.seed,

        runs=args.runs,
        time_limit=args.time_limit,
        max_trials=args.max_trials,
        trace_level=args.trace_level,
        lkh_workers=args.lkh_workers,

        linewidth=args.linewidth,
        layer_alpha=args.layer_alpha,
        dpi=args.dpi,

        save_layer_previews=(not args.no_layer_previews),
        preview_only=bool(args.preview_only),
    )

    run_sam_layers(cfg)


if __name__ == "__main__":
    main()


## 12. Notebook-specific usage example

Everything above is the original program logic, reorganized into notebook cells.

The two cells below are convenience cells for notebook use. They are not part of the original script logic, but they make the notebook easier to run interactively. Start with `preview_only=True` so you can inspect masks, density maps, and city previews before launching full LKH solves.


In [ ]:
# Example notebook usage
# Adjust these paths before running.

image_path = Path("example.jpg")
sam_ckpt = Path("sam_vit_h_4b8939.pth")
lkh_exe = Path(DEFAULT_LKH_EXE)
output_dir = Path(DEFAULT_OUTPUT_DIR)

cfg = ExperimentConfig(
    image_path=str(image_path),
    output_dir=str(output_dir),
    lkh_exe=str(lkh_exe),
    sam_ckpt=str(sam_ckpt),
    target_width=512,
    max_layers=12,
    max_cities_total=8000,
    min_cities_per_layer=250,
    max_cities_per_layer=2500,
    preview_only=True,
    nonoverlap=True,
    add_background_layer=True,
)

cfg


In [ ]:
# Run when your image, SAM checkpoint, and optional LKH executable are ready.
# run_sam_layers(cfg)
